# Clase 7 — Unidad 7: Análisis Estadístico en Biomedicina

**Curso:** Introducción al Análisis de Datos — 2026, Segundo Semestre
**Material de referencia:** *Python Essentials for Biomedical Data Analysis* — Capítulo 7, "Statistical Analysis in Biomedicine"

## Objetivos de aprendizaje
- Calcular e interpretar **estadística descriptiva**: tendencia central, variabilidad y forma de la distribución.
- Conocer las **distribuciones de probabilidad** básicas (binomial, normal) y evaluar la **normalidad**.
- Realizar **pruebas de hipótesis** (t-test y chi-cuadrado) e interpretar el **valor p**.
- Calcular e interpretar **intervalos de confianza**.
- Introducir la **correlación** y la **regresión lineal** como puente hacia el modelado.

## Contenidos de la clase
1. El rol de la estadística en la investigación biomédica
2. Estadística descriptiva
3. Distribuciones de probabilidad
4. Pruebas de hipótesis
5. Intervalos de confianza
6. Hacia el modelado: correlación y regresión
7. Ejercicios de práctica

> Esta clase corresponde a la **Unidad 7** del libro guía. Usaremos un conjunto de datos sintético con **relaciones reales inyectadas** (fumar↑ asociado a diabetes; IMC↑ asociado a glucosa) para que las pruebas estadísticas arrojen resultados interpretables.

## Preparación del entorno

Usaremos **NumPy** y **Pandas** (cálculo), **SciPy** (`scipy.stats`, el corazón de la estadística en Python) y **Matplotlib** (gráficos).

In [ ]:
%pip install numpy pandas scipy matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(7)
n = 150

edad = np.random.normal(50, 12, n).round(0)
imc = np.random.normal(27, 4, n)
glucosa = 90 + 1.3 * (imc - 27) + np.random.normal(0, 12, n)     # glucosa correlacionada con IMC
sexo = np.random.choice(["F", "M"], n)

fumador_bin = np.random.choice([1, 0], n, p=[0.35, 0.65])
prob_diab = np.where(fumador_bin == 1, 0.45, 0.15)               # fumadores: mayor prob. de diabetes
diabetes = np.random.binomial(1, prob_diab)

df = pd.DataFrame({
    "edad": edad,
    "imc": imc.round(1),
    "glucosa": glucosa.round(1),
    "sexo": sexo,
    "fumador": np.where(fumador_bin == 1, "Sí", "No"),
    "diabetes": diabetes,   # 0 = no, 1 = sí
})

# Dos grupos para comparar medias (glucosa: control vs tratamiento)
control = np.random.normal(100, 10, 60)
tratamiento = np.random.normal(94, 10, 60)

# Efecto de un fármaco (reducción de presión) para el intervalo de confianza
efecto_farmaco = np.random.normal(10, 3, 100)

AZUL = "#0072B2"
NARANJO = "#E69F00"

print("Dataset creado:", df.shape)
df.head()

## 1. El rol de la estadística en la investigación biomédica

La estadística permite **validar afirmaciones científicas**: determina si un resultado es **estadísticamente significativo** o si puede deberse al azar. Dado que los datos biológicos tienen **variabilidad natural**, los métodos estadísticos cuantifican esa incertidumbre y permiten interpretaciones más confiables.

**Aplicaciones:** diseño experimental (tamaño de muestra), evaluación de tratamientos, modelado de enfermedades y medicina personalizada.

**Desafíos:** calidad e integración de datos, alta dimensionalidad (más variables que observaciones → riesgo de *overfitting*), privacidad de datos sensibles, y la naturaleza interdisciplinaria del trabajo.

## 2. Estadística descriptiva

La estadística descriptiva resume las características clave de un conjunto de datos: **tendencia central**, **variabilidad** y **forma** de la distribución.

### 2.1 Medidas de tendencia central y variabilidad

- **Media:** promedio; útil pero sensible a outliers.
- **Mediana:** valor central; robusta ante valores extremos.
- **Moda:** valor más frecuente.
- **Rango, IQR, desviación estándar y varianza:** describen qué tan dispersos están los datos.

In [ ]:
# Ejemplo con un arreglo pequeño
datos = np.array([1, 2, 3, 4, 4, 5, 6])

print("Media:", np.mean(datos))
print("Mediana:", np.median(datos))
print("Moda:", pd.Series(datos).mode().tolist())   # pandas es robusto para la moda
print("Rango:", np.ptp(datos))                      # ptp = max - min
Q1, Q3 = np.percentile(datos, 25), np.percentile(datos, 75)
print("IQR:", Q3 - Q1)
print("Desviación estándar:", round(np.std(datos), 3))
print("Varianza:", round(np.var(datos), 3))

In [ ]:
# En un DataFrame, describe() entrega varias medidas de una sola vez
print(df[["edad", "imc", "glucosa"]].describe().round(2))

### 2.2 Forma de la distribución: asimetría (skewness) y curtosis (kurtosis)

- **Asimetría (skewness):** mide la falta de simetría. 0 = simétrica; positiva = cola a la derecha; negativa = cola a la izquierda.
- **Curtosis (kurtosis):** mide el "apuntamiento" y el peso de las colas. Cercana a 3 = similar a una normal (con `fisher=False`).

In [ ]:
from scipy.stats import skew, kurtosis

print("Glucosa -> asimetría:", round(skew(df["glucosa"]), 3),
      "| curtosis:", round(kurtosis(df["glucosa"], fisher=False), 3))

# Visualicemos la distribución de la glucosa
plt.figure(figsize=(7, 4))
plt.hist(df["glucosa"], bins=20, color=AZUL, edgecolor="white", alpha=0.85)
plt.title("Distribución de la glucosa")
plt.xlabel("Glucosa (mg/dL)")
plt.ylabel("Frecuencia")
plt.tight_layout()
plt.show()

## 3. Distribuciones de probabilidad

Una **distribución de probabilidad** describe qué tan probables son los distintos resultados de una variable aleatoria.

### 3.1 Distribución binomial (discreta)

Modela el número de "éxitos" en un número fijo de ensayos independientes. Ejemplo: ¿cuántos de 100 pacientes responden a un tratamiento, si cada uno tiene 65% de probabilidad de responder?

In [ ]:
from scipy.stats import binom

n_pacientes = 100
p_exito = 0.65

x = np.arange(0, n_pacientes + 1)
pmf = binom.pmf(x, n_pacientes, p_exito)   # función de masa de probabilidad

plt.figure(figsize=(7, 4))
plt.stem(x, pmf, linefmt=AZUL, markerfmt="o", basefmt=" ")
plt.title("Distribución binomial (PMF)")
plt.xlabel("N° de pacientes que responden")
plt.ylabel("Probabilidad")
plt.tight_layout()
plt.show()

### 3.2 Distribución normal (continua)

La **distribución normal (Gaussiana)** modela muchas mediciones biológicas que se agrupan en torno a una media con dispersión simétrica (ej. presión arterial).

In [ ]:
from scipy.stats import norm

media_pa, sd_pa = 120, 15   # presión arterial sistólica
x = np.linspace(media_pa - 4 * sd_pa, media_pa + 4 * sd_pa, 500)
pdf = norm.pdf(x, media_pa, sd_pa)   # función de densidad de probabilidad

plt.figure(figsize=(7, 4))
plt.plot(x, pdf, color=AZUL)
plt.title("Distribución normal (PDF)")
plt.xlabel("Presión arterial sistólica (mmHg)")
plt.ylabel("Densidad de probabilidad")
plt.tight_layout()
plt.show()

### 3.3 Evaluar la normalidad: test de Shapiro-Wilk

Muchas pruebas estadísticas asumen normalidad. El **test de Shapiro-Wilk** la evalúa: si el valor p es **menor a 0.05**, se rechaza la hipótesis de normalidad (los datos **no** son normales).

In [ ]:
from scipy.stats import shapiro

estadistico, p_valor = shapiro(df["glucosa"])
print(f"Shapiro-Wilk: estadístico={estadistico:.3f}, p={p_valor:.4f}")
if p_valor < 0.05:
    print("-> Se rechaza la normalidad (los datos no parecen normales).")
else:
    print("-> No se rechaza la normalidad (los datos son compatibles con una normal).")

## 4. Pruebas de hipótesis

Permiten sacar conclusiones sobre una población a partir de una muestra. Se plantean dos hipótesis:
- **H₀ (nula):** no hay efecto ni diferencia.
- **H₁ (alternativa):** sí hay efecto o diferencia.

Se fija un **nivel de significancia** α (habitualmente 0.05) y se calcula un **valor p**:
- Si **p < α** → se **rechaza H₀** (resultado significativo).
- Si **p ≥ α** → **no** se rechaza H₀.

### 4.1 Prueba t de dos muestras (comparar medias)

¿La glucosa promedio difiere entre el grupo **control** y el grupo **tratamiento**?

In [ ]:
# H0: las medias son iguales   |   H1: las medias son distintas
alpha = 0.05
t_stat, p_valor = stats.ttest_ind(control, tratamiento)

print(f"Media control: {control.mean():.1f} | Media tratamiento: {tratamiento.mean():.1f}")
print(f"Estadístico t = {t_stat:.3f}")
print(f"Valor p = {p_valor:.4f}")
if p_valor < alpha:
    print("-> Se rechaza H0: hay diferencia significativa entre los grupos.")
else:
    print("-> No se rechaza H0: no hay diferencia significativa.")

### 4.2 Prueba de chi-cuadrado (asociación entre variables categóricas)

El **chi-cuadrado** evalúa si dos variables categóricas están asociadas. Partimos de una **tabla de contingencia** (vista en la Unidad 5). ¿Están asociados el hábito de fumar y la diabetes?

In [ ]:
# Tabla de contingencia: fumador x diabetes
tabla = pd.crosstab(df["fumador"], df["diabetes"])
print("Tabla de contingencia (columnas: 0=no diabetes, 1=diabetes):")
print(tabla)

chi2, p_valor, dof, esperados = stats.chi2_contingency(tabla)
print(f"\nChi-cuadrado = {chi2:.3f} | grados de libertad = {dof} | p = {p_valor:.4f}")
if p_valor < 0.05:
    print("-> Se rechaza H0: hay asociación significativa entre fumar y diabetes.")
else:
    print("-> No se rechaza H0: no hay asociación significativa.")

## 5. Intervalos de confianza

Un **intervalo de confianza (IC)** expresa la incertidumbre de una estimación. Un IC del 95% para la media significa que, si repitiéramos el muestreo muchas veces, ~95% de los intervalos construidos contendrían la media poblacional real.

Intervalos **más angostos** → estimaciones más precisas. Calculemos el IC 95% para el efecto medio de un fármaco.

In [ ]:
media = np.mean(efecto_farmaco)
error_estandar = stats.sem(efecto_farmaco)      # error estándar de la media

# Usamos la distribución t (apropiada para muestras; con n>30 la normal da algo similar)
ic = stats.t.interval(0.95, df=len(efecto_farmaco) - 1, loc=media, scale=error_estandar)

print(f"Efecto medio estimado: {media:.2f}")
print(f"IC 95%: [{ic[0]:.2f}, {ic[1]:.2f}]")

## 6. Hacia el modelado: correlación y regresión

### 6.1 Correlación

La **correlación de Pearson** cuantifica la fuerza y dirección de la relación **lineal** entre dos variables numéricas (de −1 a 1).

In [ ]:
from scipy.stats import pearsonr

r, p_valor = pearsonr(df["imc"], df["glucosa"])
print(f"Correlación de Pearson (IMC vs glucosa): r = {r:.3f}, p = {p_valor:.4g}")

### 6.2 Regresión lineal simple

La **regresión lineal** modela una variable dependiente en función de una predictora. Aquí predecimos la glucosa a partir del IMC. `linregress` devuelve la pendiente, el intercepto, el coeficiente r y el valor p.

In [ ]:
resultado = stats.linregress(df["imc"], df["glucosa"])
print(f"Pendiente: {resultado.slope:.2f}  (glucosa sube por cada punto de IMC)")
print(f"Intercepto: {resultado.intercept:.2f}")
print(f"R²: {resultado.rvalue**2:.3f}  |  p = {resultado.pvalue:.4g}")

# Gráfico: dispersión + recta ajustada
plt.figure(figsize=(7, 4))
plt.scatter(df["imc"], df["glucosa"], alpha=0.5, color=AZUL, label="Datos")
x_linea = np.array([df["imc"].min(), df["imc"].max()])
plt.plot(x_linea, resultado.intercept + resultado.slope * x_linea,
         color=NARANJO, linewidth=2, label="Recta ajustada")
plt.title("Regresión lineal: IMC → glucosa")
plt.xlabel("IMC (kg/m²)")
plt.ylabel("Glucosa (mg/dL)")
plt.legend()
plt.tight_layout()
plt.show()

> **Métodos avanzados (próximas unidades):** la **regresión logística** modela resultados binarios (ej. enfermedad sí/no); el **análisis multivariado** (PCA, análisis factorial) estudia muchas variables a la vez; y el **análisis de supervivencia** (Kaplan-Meier, Cox) modela el tiempo hasta un evento. Los veremos con más detalle en la unidad de *machine learning*.

## 7. Ejercicios de práctica

Usa el conjunto `df` (o crea tus propios datos) para resolver:

1. **Descriptiva:** calcula media, mediana, desviación estándar e IQR de la columna `edad`.
2. **Forma:** calcula la asimetría y la curtosis de `imc`. ¿La distribución es simétrica?
3. **Normalidad:** aplica el test de Shapiro-Wilk a `imc` e interpreta el valor p.
4. **t-test:** compara la glucosa promedio entre fumadores y no fumadores (`df` filtrado por `fumador`). ¿Hay diferencia significativa?
5. **Chi-cuadrado:** construye la tabla de contingencia `sexo` × `diabetes` y evalúa si están asociados.
6. **Intervalo de confianza:** calcula el IC 95% para la media de `glucosa`.
7. **Regresión (desafío):** ajusta una regresión lineal de `edad` (predictora) sobre `glucosa` y grafica la recta ajustada. ¿La edad predice bien la glucosa?

### Preguntas para reflexionar
1. ¿Qué significa un valor p menor a 0.05? ¿Y qué **no** significa?
2. ¿Cuándo conviene usar la mediana en lugar de la media?
3. ¿Por qué se usa un t-test para comparar medias y un chi-cuadrado para variables categóricas?
4. ¿Qué diferencia hay entre una correlación fuerte y una relación causal?
5. Un IC 95% más angosto, ¿indica mayor o menor precisión en la estimación?

In [ ]:
# Espacio para resolver los ejercicios
